# Clase 201 — Serverless ML: AWS Lambda + GCP Cloud Functions

Notebook **declarativo + cost calculator**. Genera el Dockerfile/handler para Lambda y el script `main.py` para Cloud Functions, y simula el costo mensual de cada arquitectura.

## 1. Lambda Container Image

In [ ]:
lambda_dockerfile = '''\
FROM public.ecr.aws/lambda/python:3.12

COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py model.pkl ./

CMD ["app.lambda_handler"]
'''
print('# Dockerfile (Lambda)')
print(lambda_dockerfile)

In [ ]:
lambda_handler = '''\
import json, joblib, numpy as np

# Module-level: corre UNA vez por instancia (cold start), no por request.
MODEL = joblib.load("/var/task/model.pkl")

def lambda_handler(event, context):
    body = event.get("body")
    if isinstance(body, str):
        body = json.loads(body)
    features = body.get("features", [])
    if len(features) != 4 or any(x < 0 for x in features):
        return {"statusCode": 422, "body": json.dumps({"error": "invalid features"})}
    pred = int(MODEL.predict(np.asarray(features).reshape(1, -1))[0])
    return {"statusCode": 200, "body": json.dumps({"class": pred}),
            "headers": {"content-type": "application/json"}}
'''
print('# app.py (Lambda handler)')
print(lambda_handler)

In [ ]:
lambda_deploy = '''\
# Build + push + create
REGION=us-east-1
ACCOUNT=$(aws sts get-caller-identity --query Account --output text)
REPO=iris-lambda

aws ecr create-repository --repository-name $REPO --region $REGION || true
aws ecr get-login-password --region $REGION | docker login --username AWS \\
    --password-stdin $ACCOUNT.dkr.ecr.$REGION.amazonaws.com

docker build -t $REPO:v1 .
docker tag $REPO:v1 $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/$REPO:v1
docker push $ACCOUNT.dkr.ecr.$REGION.amazonaws.com/$REPO:v1

aws lambda create-function \\
    --function-name iris-predict \\
    --package-type Image \\
    --code ImageUri=$ACCOUNT.dkr.ecr.$REGION.amazonaws.com/$REPO:v1 \\
    --role arn:aws:iam::$ACCOUNT:role/lambda-exec \\
    --timeout 30 --memory-size 512

# Mitigar cold start (opcional, cuesta):
aws lambda put-provisioned-concurrency-config \\
    --function-name iris-predict --qualifier 1 \\
    --provisioned-concurrent-executions 2
'''
print(lambda_deploy)

## 2. Cloud Functions 2nd gen

In [ ]:
gcf_main = '''\
# main.py
import functions_framework, joblib, numpy as np

MODEL = joblib.load("model.pkl")   # cargado 1 vez por instancia

@functions_framework.http
def predict(request):
    data = request.get_json(silent=True) or {}
    features = data.get("features", [])
    if len(features) != 4 or any(x < 0 for x in features):
        return ({"error": "invalid features"}, 422)
    pred = int(MODEL.predict(np.asarray(features).reshape(1, -1))[0])
    return {"class": pred}
'''
print('# main.py + requirements.txt')
print(gcf_main)

gcf_deploy = '''\
gcloud functions deploy iris-predict \\
    --gen2 --runtime=python312 --region=us-central1 \\
    --source=. --entry-point=predict \\
    --trigger-http --allow-unauthenticated \\
    --memory=512Mi --timeout=60s \\
    --min-instances=1 --max-instances=20
'''
print(gcf_deploy)

## 3. Cost calculator — Lambda vs K8s

Precios aproximados (junio 2026, us-east-1):
- Lambda on-demand: $0.20 / M requests + $0.0000166667 / GB-s
- Lambda Provisioned Concurrency: $0.0000041667 / GB-s (always-on) + above per-request
- t3.medium on-demand (4 GB, 2 vCPU): $0.0416/h ≈ $30/mes

In [ ]:
def cost_lambda(rps, dur_ms, mem_mb, days=30, pc=0):
    """Lambda monthly cost in USD. pc = provisioned concurrency instances."""
    invocs = rps * 86400 * days
    gb_s = invocs * (dur_ms / 1000) * (mem_mb / 1024)
    request_cost = invocs / 1_000_000 * 0.20
    duration_cost = gb_s * 0.0000166667
    pc_cost = pc * (mem_mb / 1024) * 86400 * days * 0.0000041667 if pc else 0
    return request_cost + duration_cost + pc_cost

def cost_k8s(pods=3, instance_usd_month=30):
    return pods * instance_usd_month

scenarios = [
    ('bursty 1 req/s mean', 1, 80, 512),
    ('moderado 10 req/s', 10, 80, 512),
    ('alto 100 req/s', 100, 80, 512),
    ('muy alto 1000 req/s', 1000, 80, 512),
]
print(f'{"scenario":30} {"Lambda":>10} {"Lambda+PC=2":>14} {"K8s 3pods":>12}')
for name, rps, dur, mem in scenarios:
    l = cost_lambda(rps, dur, mem)
    l_pc = cost_lambda(rps, dur, mem, pc=2)
    k = cost_k8s(3)
    print(f'{name:30} ${l:>9.0f} ${l_pc:>13.0f} ${k:>11.0f}')

print('\n→ Lambda gana a baja carga; K8s gana cuando rps sostenido es alto.')

## Ejercicio guiado

1. Deployá la Lambda real (requiere cuenta AWS). Medí `InitDuration` en CloudWatch — primera invocación cold, siguientes warm.
2. Activá Provisioned Concurrency=2. Repetí test. Compará p99 latency con/sin PC. Compará costo.
3. Deployá la Cloud Function equivalente. Compará cold start con Lambda (suele ser similar en 2nd gen).
4. Modificá `cost_lambda` para sumar API Gateway ($1/M requests). Calculá el punto donde Lambda Function URL (gratis) cambia la economía.
5. Para tu caso real (estimá `rps`, `dur_ms`, `mem_mb`): decidí Lambda vs ECS Fargate vs K8s.

## Conclusiones

- Serverless gana en: tráfico bursty, equipo chico, modelo <1 GB, latencia tolerante.
- Cold start es el villano — mitigá con module-level init + (opcional) PC/min-instances.
- Punto de cruce con K8s: ~100-500 req/s sostenido (depende del modelo y región).
- Container Image hasta 10 GB es el único formato razonable para ML hoy.